<a href="https://colab.research.google.com/github/ChitrashreeShankaranandha/AgenticAI_LLMs_RAG/blob/main/RAG_LLM_PDF_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Introduction**

Retrieval-augmented generation (RAG) enabled the use of pre-trained models by providing the right context to the LLM.

Providing the right context from a knowledge source (e.g., a set of documents) allows:
*   Incorporating domain-specific knowledge into the LLM.
*   Reducing LLM hallucination by grounding responses in factual data.
*   Answering questions based on private or up-to-date information not present in the LLM's original training data.

# **Problem Statement**

Imagine you're a student juggling multiple subjects, each with extensive lecture notes (provided
as text files or PDFs). It's challenging to quickly find specific information or revise concepts
across these varied documents. You need a "Smart Study Buddy" – a virtual assistant that can:

*   Understand your questions about the course material.

*   Retrieve relevant information only from your uploaded lecture notes.

*   Maintain the context of your current study session, allowing for follow-up questions.


This tool aims to make studying more efficient and targeted.

**ML Problem**

To build the "Smart Study Buddy," we will develop a Retrieval Augmented Generation (RAG)
system. This system will:

* Data Sources (Documents): Ingest multiple local document files (e.g., .txt, .pdf) representing your lecture notes.

* Vector Store: Use Pinecone to store a searchable representation (embeddings) of your lecture notes. This allows for efficient similarity search to find relevant context.
* LLM: Leverage a Large Language Model (LLM) to understand questions, process
retrieved context, and generate helpful answers.
* Conversational Memory (In-Session): Implement a mechanism to remember the
dialogue within the current study session, enabling coherent follow-up interactions.

**Project Tasks**

The project has been broken down into the following key tasks:
1. Pinecone Setup
  * You will need a Pinecone account.
  * Initialize the Pinecone client in your notebook using your API key and environment.
  * Define a unique name for your Pinecone index.
  * Write code to create the index if it doesn't already exist. Ensure you specify the correct
  dimension for the embedding model you choose (e.g., OpenAI's
  text-embedding-ada-002 has a dimension of 1536).
  * Reference: Pinecone LangChain Integration

2. Data Ingestion and Processing (Loading and Chunking)
  * Create 2-3 sample lecture note files (e.g., subjectA_notes.txt,
  subjectB_notes.pdf). You can fill them with dummy text or copy-paste some
  educational content.
  * Use appropriate LangChain document loaders to load the content from these files.

     - TextLoader for .txt files.
     - PyPDFLoader for .pdf files. (You might need to pip install pypdf)
  
  * Chunk the loaded documents into smaller, manageable pieces. This is crucial for fitting
  within the LLM's context window and for effective retrieval.
    - Consider using RecursiveCharacterTextSplitter or CharacterTextSplitter.
    - Experiment with chunk_size and chunk_overlap.

3. Embedding Generation and Vector Storage
* For each document chunk, generate an embedding (a numerical vector representation).

* You can use:
  - OpenAIEmbeddings (requires an OpenAI API key).
  - Sentence Transformer models from Hugging Face via
  HuggingFaceEmbeddings (e.g., 'all-MiniLM-L6-v2').

* Store these chunks and their corresponding embeddings in your Pinecone index.
  - Use PineconeVectorStore.from_documents() for an easy way to
  populate the index.
  - Ensure the embedding model used here matches the dimension specified during
  Pinecone index creation.

4. Conversational Question Answering Chain
* Implement a chain that can answer questions based on the retrieved context and
maintain conversation history for the current session.
* ConversationalRetrievalChain is a good high-level option.
* Alternatively, you can build a custom chain using a retriever from your Pinecone vector
store and integrating a memory module.
* Use a LangChain memory module like ConversationBufferMemory or
ConversationSummaryMemory to store the chat history of the current session.
* Reference: Adding Memory to QA Chains
* Create a PromptTemplate to guide the LLM. The prompt should instruct the LLM to:
- Answer the user's question based only on the provided context (retrieved
chunks).
- If the answer cannot be found in the provided context, explicitly state that the
information is not in the lecture notes.
- Consider the chat history when formulating the current answer.
- Reference: PromptTemplate

5. Testing and Demonstration
* In your notebook, instantiate and interact with your "Smart Study Buddy."
* Test Case 1: Ask an initial question whose answer is clearly in one of your documents.
* Test Case 2: Ask a follow-up question that relies on the context established by the
previous question-answer pair.

* Example:
  - User: "What are the main types of RAG systems?"
  - Bot: "Type A, Type B, Type C"

  - User: "Tell me more about Type A."

* Test Case 3: Ask a question for which the answer is not present in your documents to
verify the "I don't know" or "not in notes" behavior.
* Test Case 4 (Optional): Restart your notebook kernel and run the interaction part again.
The chatbot should not remember the conversation from the previous run (as we are not
implementing persistent user memory), but it should still be able to answer questions
based on the documents already indexed in Pinecone.

Extra Requirements (Focus on In-Session Conversation)
The core requirement is to make the system conversational *within a single interaction session*.
The system should understand follow-up questions that implicitly refer to previous turns in the
current dialogue.

**Example Interaction:**
* User: "What were the key topics covered in the Machine Learning lecture?"
* System: (Retrieves from notes) "The key topics included Supervised Learning,
Unsupervised Learning, and Reinforcement Learning."
* User: "Can you explain Supervised Learning in more detail based on the notes?"
- (The system should understand "Supervised Learning" refers to the one
mentioned in its previous response and use the notes to elaborate.)

This is achieved using LangChain's memory components, ensuring the `chat_history` is passed
appropriately.

In [ ]:
!pip -q uninstall -y langchain-classic langgraph-prebuilt google-adk || true
!pip -q uninstall -y langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-pinecone || true

In [ ]:
!pip -q install \
  "langchain==0.2.17" \
  "langchain-core==0.2.43" \
  "langchain-community==0.2.16" \
  "langchain-openai==0.1.23" \
  "langchain-text-splitters==0.2.4" \
  "langchain-pinecone==0.1.2" \
  "pinecone-client==3.2.2" \
  "openai>=1.0.0" \
  pypdf python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.0.8 requires langgraph-prebuilt<1.1.0,>=1.0.7, which is not installed.


In [ ]:
from google.colab import userdata

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

# Load keys from Colab Secrets (these must match your Secrets panel names)
PINECONE_API_KEY = userdata.get("PineConeA")
OPENAI_API_KEY = userdata.get("OpenAIApi")

if not PINECONE_API_KEY:
    raise ValueError("Missing Colab Secret: PineConeA")
if not OPENAI_API_KEY:
    raise ValueError("Missing Colab Secret: OpenAIApi")

print("✅ Keys loaded")

✅ Keys loaded


In [ ]:
INDEX_NAME = "study-buddy"
NAMESPACE = "lecture_ml_01_2024"

embeddings = OpenAIEmbeddings(
    api_key=OPENAI_API_KEY,
    model="text-embedding-3-small"  # 1536 dims
)

vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    pinecone_api_key=PINECONE_API_KEY,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("✅ Connected to Pinecone:", INDEX_NAME, "| namespace:", NAMESPACE)

✅ Connected to Pinecone: study-buddy | namespace: lecture_ml_01_2024


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

docs = PyPDFLoader("/content/lecture_ml_01_2024.pdf").load()

print("✅ Loaded pages:", len(docs))
print(docs[0].page_content[:300])

✅ Loaded pages: 75
Machine Learning
Introduction. ML History
Aleksandr Petiushko
ML Research
A. Petiushko Intro. ML History 1 / 24


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
splits = splitter.split_documents(docs)
print("✅ Chunks:", len(splits))

vectorstore.add_documents(splits)

print("✅ Upserted chunks to Pinecone (namespace):", NAMESPACE)

✅ Chunks: 118
✅ Upserted chunks to Pinecone (namespace): lecture_ml_01_2024


In [ ]:
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",
    temperature=0
)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True
)

print("✅ Conversational RAG chain ready")

✅ Conversational RAG chain ready


In [ ]:
def ask(q):
    out = qa({"question": q})
    print("Q:", q)
    print("A:", out["answer"])
    print("\nSources:")
    for d in out["source_documents"][:2]:
        print("-", d.metadata.get("source", "unknown"))
    print("-"*80)

ask("What is supervised learning?")
ask("Explain it in simple terms using the lecture.")
ask("List 3 key topics covered in this lecture.")

# hallucination test (likely not in the lecture)
ask("What is the capital of France?")

Q: What is supervised learning?
A: Supervised learning is a machine learning paradigm where a sufficient amount of training material is provided, consisting of pairs of input data (xi) and their corresponding correct answers or labels (yi). The goal is to learn a function that maps the input data to the correct outputs based on this training data.

Sources:
- /content/lecture_ml_01_2024.pdf
- /content/lecture_ml_01_2024.pdf
--------------------------------------------------------------------------------
Q: Explain it in simple terms using the lecture.
A: Supervised learning is a type of machine learning where a model is trained using a set of data that includes both the input (features) and the correct output (labels). In simple terms, it's like teaching a child with examples: you show them a picture of a dog and tell them, "This is a dog." Over time, with enough examples, the child learns to recognize dogs on their own. Similarly, in supervised learning, the model learns to make predi